### Set Up

In [22]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [23]:
import json
import zipfile
import tempfile
import os
import shutil
import numpy as np
import keras
import torch
import pandas as pd
from PIL import Image
from torchvision import transforms

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

keras.mixed_precision.set_global_policy("mixed_float16")
keras.config.set_image_data_format("channels_first")

print("Device:", device)

Device: cuda


### Load Model

In [25]:
weights_path = "model/alexnet_fp16.weights.h5"
zip_path = "model/alexnet_fp16_weights.zip"

In [26]:
@keras.saving.register_keras_serializable()
class LocalResponseNormalization(keras.layers.Layer):
    def call(self, x):
        return torch.nn.functional.local_response_norm(x, size=5, alpha=1e-4, beta=0.75, k=2.0, )

In [27]:
def load_int8_model(arch_path, zip_path):
    with open(arch_path, "r") as f:
        model = keras.models.model_from_json(f.read(), custom_objects={"LocalResponseNormalization": LocalResponseNormalization})

    temp_dir = tempfile.mkdtemp()
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(temp_dir)

        npz_path = os.path.join(temp_dir, "alexnet_int8_weights.npz")
        meta_path = os.path.join(temp_dir, "alexnet_int8_metadata.json")

        with open(meta_path, "r") as f:
            metadata = json.load(f)

        with np.load(npz_path) as data:
            weight_arrays = {k: data[k].copy() for k in data.files}

        for i, layer in enumerate(model.layers):
            original_weights = layer.get_weights()
            if not original_weights:
                continue
            new_weights = []
            for j in range(len(original_weights)):
                key = f"layer_{i}_weight_{j}"
                int8_w = weight_arrays[key]
                scale = metadata[key]["scale"]
                float_w = int8_w.astype(np.float32) * scale
                new_weights.append(float_w)
            layer.set_weights(new_weights)

    finally:
        try:
            shutil.rmtree(temp_dir, ignore_errors=True)
        except Exception as e:
            print(f"Warning: could not clean temp dir {temp_dir}: {e}")
    return model

In [28]:
model = load_int8_model("model/alexnet_architecture.json", "model/alexnet_int8_weights.zip")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 3, 224, 224)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d (ZeroPadding2D)  │ (None, 3, 228, 228)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 96, 55, 55)     │        34,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lrn1                            │ (None, 96, 55, 55)     │             0 │
│ (LocalResponseNormalization)    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 96, 27, 27)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 256, 27, 27)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lrn2                            │ (None, 256, 27, 27)    │             0 │
│ (LocalResponseNormalization)    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 256, 13, 13)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 384, 13, 13)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4 (Conv2D)                  │ (None, 384, 13, 13)    │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv5 (Conv2D)                  │ (None, 256, 13, 13)    │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool5 (MaxPooling2D)            │ (None, 256, 6, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 4096)           │    37,752,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense2 (Dense)                  │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense3 (Dense)                  │ (None, 1000)           │     4,097,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,378,344 (237.95 MB)

 Trainable params: 62,378,344 (237.95 MB)

 Non-trainable params: 0 (0.00 B)

### Utils

In [29]:
class_map_file = "../../Datasets/ILSVRC2010_images/IDX_WNID_CLASS.csv"
class_map = pd.read_csv(class_map_file)
idx_to_wnid = dict(zip(class_map["idx"], class_map["wnid"]))
wnid_to_class = dict(zip(class_map["wnid"], class_map["class_name"]))

In [30]:
def get_class_name(target_id):
    return wnid_to_class[idx_to_wnid[target_id]]

In [31]:
def get_class_name(target_id):
    return wnid_to_class[idx_to_wnid[target_id]]

In [32]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

### Load Image

In [34]:
image_file = "test_image.JPEG"

In [35]:
image = Image.open(image_file).convert("RGB")
x = preprocess(image)
x = x.unsqueeze(0)
x = x.to(device)

### inference

In [36]:
with torch.inference_mode():
    predictions = model(x)
    predictions = predictions.float()
    predictions = predictions.cpu().numpy()

In [37]:
predicted_class = get_class_name(np.argmax(predictions[0]))
predicted_class

'airliner'

In [38]:
top5 = np.argsort(predictions[0])[-5:][::-1]

for idx in top5:

    print(
        f"{get_class_name(idx):30s} | "
        f"{100 * predictions[0][idx]:.2f}%"
    )

airliner                       | 86.08%
warplane, military plane       | 9.66%
airship, dirigible             | 2.12%
missile                        | 0.60%
submarine, pigboat, sub, U-boat | 0.45%


end